In [5]:
import glob
import numpy as np
import os
import pandas as pd

from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

folder_path = '../data/'
result_path = folder_path + 'results/'
annotation_path = result_path + 'datasets_annotation/'

In [2]:
CATEGORIES = ['AQA-ready', 'AQA-Related', 'Not Applicable to AQA']

def parse_annotator_file(filepath, rater_name):
    df = pd.read_excel(filepath, skiprows=2)
    df = df.rename(columns={df.columns[0]: 'Dataset Name'})
    df_labels = df.iloc[:, 3:6]

    def get_label(row):
        row_vals = row.fillna(0).values
        if row_vals.max() > 0:
            return CATEGORIES[np.argmax(row_vals)]
        return np.nan

    df[rater_name] = df_labels.apply(get_label, axis=1)

    return df[['Dataset Name', rater_name]].dropna(subset=['Dataset Name'])

In [4]:
file_pattern = os.path.join(annotation_path, 'datasets_annotation_*.xlsx')
annotator_files = glob.glob(file_pattern)

master_df = None

for i, filepath in enumerate(annotator_files):
    rater_name = f'Rater_Ext_{i+1}'
    rater_df = parse_annotator_file(filepath, rater_name)

    if master_df is None:
        master_df = rater_df
    else:
        master_df = pd.merge(master_df, rater_df, on='Dataset Name', how='outer')

In [7]:
print("--- Summary Table (Master Table) ---")
print(master_df.head())
print("------------------------------------")

rater_columns = master_df.columns[1:]
ratings_matrix = master_df[rater_columns].values

agg_data, categories_found = aggregate_raters(ratings_matrix)

# Fleiss' Kappa
kappa = fleiss_kappa(agg_data, method='fleiss')

print(f"\nInter-Rater Reliability Results:")
print(f"Number of analyzed datasets: {len(master_df)}")
print(f"Number of raters: {len(rater_columns)}")
print(f"Categories detected: {categories_found}")
print(f"Fleiss' Kappa: {kappa:.4f}")

if kappa > 0.8:
    print("Agreement: Almost Perfect (Excellent)")
elif kappa > 0.6:
    print("Agreement: Substantial")
elif kappa > 0.4:
    print("Agreement: Moderate")
else:
    print("Agreement: Fair/Poor")

print("\n--- Disagreements Overview ---")
disagreements_df = master_df[master_df[rater_columns].nunique(axis=1) > 1]

if disagreements_df.empty:
    print("No disagreements found! All raters are in perfect agreement.")
else:
    print(f"Found {len(disagreements_df)} datasets with disagreements:\n")
    print(disagreements_df.to_string(index=False))
print("------------------------------")

--- Summary Table (Master Table) ---
     Dataset Name  Rater_Ext_1  Rater_Ext_2
0    3D Animation    AQA-ready    AQA-ready
1    AGF-Olympics    AQA-ready    AQA-ready
2  ActivityNet1.3  AQA-Related  AQA-Related
3          CAD-60  AQA-Related  AQA-Related
4       CPR-Coach    AQA-ready    AQA-ready
------------------------------------

Inter-Rater Reliability Results:
Number of analyzed datasets: 32
Number of raters: 2
Categories detected: ['AQA-Related' 'AQA-ready']
Fleiss' Kappa: 0.8123
Agreement: Almost Perfect (Excellent)

--- Disagreements Overview ---
Found 3 datasets with disagreements:

     Dataset Name Rater_Ext_1 Rater_Ext_2
Perturbed Workout   AQA-ready AQA-Related
     SkatingVerse   AQA-ready AQA-Related
        TGC20ReId   AQA-ready AQA-Related
------------------------------
